In [1]:
# --- 1. Import Libraries ---
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# --- 2. Load Dataset ---
df = pd.read_csv("cleaned_croma_laptops.csv")

# --- 3. Features & Target ---
X = df.drop("Price", axis=1)   # predictors
y = df["Price"]                # target (price)

# --- 4. Train-Test Split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# --- 5. Identify Categorical & Numeric Features ---
categorical_features = X.select_dtypes(include=["object"]).columns
numeric_features = X.select_dtypes(include=[np.number]).columns

# --- 6. Preprocessing ---
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

# --- 7. Build Decision Tree Model ---
dt_model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", DecisionTreeRegressor(
        max_depth=None,   # allow tree to grow fully
        random_state=42
    ))
])

# --- 8. Train Model ---
dt_model.fit(X_train, y_train)

# --- 9. Predictions ---
y_pred = dt_model.predict(X_test)

# --- 10. Evaluation Metrics ---
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Decision Tree Regression Results")
print("MAE :", mae)
print("RMSE:", rmse)
print("R²  :", r2)


Decision Tree Regression Results
MAE : 17493.42536340852
RMSE: 29333.76235669874
R²  : 0.7981823124897539


In [2]:
import pickle

# Save the trained Ridge model
with open("decision_model.pkl", "wb") as file:
    pickle.dump(dt_model, file)